# 06a — QLoRA fine-tuning (9 runs)

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**Phase 6 goal:** **QLoRA** = LoRA on a base model that's been compressed to 4-bit. The research question: *does quantising the base hurt accuracy?* If accuracy holds while memory drops, QLoRA wins as the practical recipe.

**This notebook (06a):** runs **9 QLoRA training jobs** — for each of the three models, apply its best Phase-5 LoRA hyperparameters to BBBP, BACE, and ESOL.

**Best configs from Phase 5 (winners from the BBBP sweeps):**

```
Phi-4-mini-instruct  : rank=8,  lr=1e-4
SmolLM3-3B           : rank=8,  lr=5e-4
GPT-2                : rank=16, lr=5e-4
```

**Supervisor's spec for Phase 6 (lines 91–96):**
- ✅ Load base in 4-bit NF4 via `BitsAndBytesConfig`
- ✅ Same LoRA settings as Phase 5
- ✅ Optimizer: `paged_adamw_8bit`
- ✅ BBBP → BACE → ESOL with the chosen config
- ✅ Append to results table

**06b will handle the supervisor's comparison ask** (accuracy/memory/time/inference-latency deltas vs Phase 5).

**Estimated runtime on T4:** ~60–90 min. QLoRA is slightly slower than fp16 LoRA per step (4-bit dequantisation overhead), but uses far less memory.

## 1. Install + imports

In [ ]:
# Same torchao-first install pattern as Phase 5
%pip install -q --upgrade torchao
%pip install -q --upgrade transformers peft accelerate bitsandbytes rdkit pandas scikit-learn

In [ ]:
import os, random, gc, time
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import bitsandbytes as bnb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")

## 2. Mount Drive + paths

QLoRA results go to a **separate CSV** (`qlora_results.csv`) so we can diff cleanly against `lora_results.csv` in 06b.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR     = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR  = "/content/drive/MyDrive/chemistry-peft-fyp/results"
ADAPTERS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/adapters"
QLORA_CSV    = f"{RESULTS_DIR}/qlora_results.csv"
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
print("Adapters dir   :", ADAPTERS_DIR)
print("QLoRA results  :", QLORA_CSV)

## 3. (Optional) HF login

Only needed if Phi-4-mini is gated on your account. Uncomment if section 8 returns 401.

In [ ]:
# from huggingface_hub import login
# login()

## 4. Splits + dataset configs + helpers (same as Phase 5)

In [ ]:
def load_splits(name):
    p = name.lower()
    return (
        pd.read_csv(f"{DATA_DIR}/{p}_train.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_val.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_test.csv"),
    )

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}
for n, (tr, va, te) in splits.items():
    print(f"{n:5s}: train={len(tr)}  val={len(va)}  test={len(te)}")

ESOL_BUCKETS = [
    ("very_low",  -np.inf, -6.0, -7.5),
    ("low",       -6.0,    -4.0, -5.0),
    ("medium",    -4.0,    -2.0, -3.0),
    ("high",      -2.0,     0.0, -1.0),
    ("very_high",  0.0,    np.inf, 1.0),
]
ESOL_LABEL_WORDS = [b[0] for b in ESOL_BUCKETS]
ESOL_MIDPOINTS   = np.array([b[3] for b in ESOL_BUCKETS], dtype=np.float32)

def esol_continuous_to_bucket(val):
    for label, lo, hi, _ in ESOL_BUCKETS:
        if lo < val <= hi: return label
    return ESOL_BUCKETS[0][0]

@dataclass
class DatasetConfig:
    name: str; task: str; question: str
    label_words: List[str]; positive_word: Optional[str]

DATASETS = {
    "BBBP": DatasetConfig("BBBP", "classification",
        "Does this molecule cross the blood-brain barrier?",
        ["yes", "no"], "yes"),
    "BACE": DatasetConfig("BACE", "classification",
        "Does this molecule inhibit the BACE-1 enzyme?",
        ["yes", "no"], "yes"),
    "ESOL": DatasetConfig("ESOL", "regression",
        "What is the aqueous solubility of this molecule?",
        ESOL_LABEL_WORDS, None),
}

def label_for_prompt(cfg, raw_label):
    if cfg.task == "classification":
        return cfg.positive_word if int(raw_label) == 1 else "no"
    return esol_continuous_to_bucket(float(raw_label))

def build_prompt(cfg, test_smiles):
    return f"{cfg.question}\nSMILES: {test_smiles}\nAnswer:"

IGNORE_INDEX = -100
MAX_LEN = 256

class PromptAnswerDataset(TorchDataset):
    def __init__(self, df, cfg, tokenizer):
        self.cfg = cfg; self.tok = tokenizer
        self.rows = []
        for _, row in df.iterrows():
            prompt = build_prompt(cfg, row["smiles"])
            answer = label_for_prompt(cfg, row["label"])
            self.rows.append((prompt, answer, row["label"]))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        prompt, answer, _ = self.rows[i]
        prompt_ids = self.tok(prompt, add_special_tokens=False).input_ids
        answer_ids = self.tok(" " + answer, add_special_tokens=False).input_ids
        eos = [self.tok.eos_token_id] if self.tok.eos_token_id is not None else []
        input_ids = (prompt_ids + answer_ids + eos)[:MAX_LEN]
        labels    = ([IGNORE_INDEX] * len(prompt_ids) + answer_ids + eos)[:MAX_LEN]
        attention = [1] * len(input_ids)
        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
            "attention_mask": torch.tensor(attention,  dtype=torch.long),
        }

def collate_pad(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    def pad_to(t, val):
        return torch.cat([t, torch.full((max_len - len(t),), val, dtype=t.dtype)])
    return {
        "input_ids":      torch.stack([pad_to(b["input_ids"],      pad_id) for b in batch]),
        "labels":         torch.stack([pad_to(b["labels"],         IGNORE_INDEX) for b in batch]),
        "attention_mask": torch.stack([pad_to(b["attention_mask"], 0) for b in batch]),
    }

print("Helpers ready.")

## 5. Model configurations + Phase-5 best hyperparameters

Each model carries its own best `(rank, lr)` from the Phase-5 BBBP sweep, plus the right LoRA target modules and a sensible batch size for QLoRA on T4.

In [ ]:
MODEL_CONFIGS = {
    "microsoft/Phi-4-mini-instruct": {
        "short_name":             "phi4",
        "target_modules":         ["qkv_proj", "o_proj"],
        "best_rank":              8,
        "best_lr":                1e-4,
        "batch_size":             2,
        "gradient_checkpointing": True,
    },
    "HuggingFaceTB/SmolLM3-3B": {
        "short_name":             "smolllm3",
        "target_modules":         ["q_proj", "v_proj"],
        "best_rank":              8,
        "best_lr":                5e-4,
        "batch_size":             2,
        "gradient_checkpointing": True,
    },
    "gpt2": {
        "short_name":             "gpt2",
        "target_modules":         ["c_attn"],
        "best_rank":              16,
        "best_lr":                5e-4,
        "batch_size":             8,
        "gradient_checkpointing": False,
    },
}

for mid, cfg in MODEL_CONFIGS.items():
    print(f"  {mid:35s}  rank={cfg['best_rank']:2d}  lr={cfg['best_lr']}")

## 6. The QLoRA training/eval function

The differences vs the Phase-5 `lora_train_eval`:
1. **`BitsAndBytesConfig`** wraps the base model in 4-bit NF4 with double quantisation (the standard QLoRA recipe).
2. **`prepare_model_for_kbit_training`** casts non-quantised layers (LayerNorm, embeddings) to fp32 for stable training, and enables input-grad propagation through the quantised weights.
3. **`bnb.optim.PagedAdamW8bit`** replaces vanilla AdamW — supervisor's exact ask. 8-bit optimizer states + GPU-page-out support.
4. All other settings (LoRA target modules, rank, lr, epochs, dataset, eval) are identical to Phase 5 — so the comparison is clean.

In [ ]:
@torch.no_grad()
def score_labels(model, tokenizer, prompt, label_words):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    out = model(**enc)
    next_logits = out.logits[0, -1, :]
    log_probs = torch.log_softmax(next_logits, dim=-1)
    token_ids_per_label = []
    for w in label_words:
        cands = set()
        for variant in (" " + w, w):
            t = tokenizer(variant, add_special_tokens=False).input_ids
            if t: cands.add(t[0])
        token_ids_per_label.append(sorted(cands))
    label_logps = [max(log_probs[i].item() for i in ids) for ids in token_ids_per_label]
    logps = np.array(label_logps, dtype=np.float64)
    logps -= logps.max()
    probs = np.exp(logps); probs /= probs.sum()
    return probs

def eval_model(model, tokenizer, dataset_name):
    cfg = DATASETS[dataset_name]
    _, _, te_df = splits[dataset_name]
    probs_all, y_true = [], []
    model.eval()
    for _, row in te_df.iterrows():
        probs_all.append(score_labels(model, tokenizer, build_prompt(cfg, row["smiles"]), cfg.label_words))
        y_true.append(row["label"])
    probs_all = np.stack(probs_all); y_true = np.array(y_true)
    if cfg.task == "classification":
        pos_idx = cfg.label_words.index(cfg.positive_word)
        p_pos = probs_all[:, pos_idx]
        pred  = (p_pos >= 0.5).astype(int)
        return {"primary_metric": "roc_auc",
                "primary_value":  float(roc_auc_score(y_true, p_pos)),
                "secondary_metric": "f1",
                "secondary_value":  float(f1_score(y_true, pred))}
    preds_cont = (probs_all * ESOL_MIDPOINTS[None, :]).sum(axis=1)
    y_cont = y_true.astype(np.float32)
    return {"primary_metric": "rmse",
            "primary_value":  float(np.sqrt(mean_squared_error(y_cont, preds_cont))),
            "secondary_metric": "mae",
            "secondary_value":  float(mean_absolute_error(y_cont, preds_cont))}

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def qlora_train_eval(model_id, dataset_name, epochs=3, save_adapter_to=None):
    mc = MODEL_CONFIGS[model_id]
    cfg = DATASETS[dataset_name]
    tr_df, _, _ = splits[dataset_name]

    print(f"\nLoading {model_id} in 4-bit (NF4 + double quant)...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BNB_CONFIG,
        device_map="auto",
    )
    # Prep for k-bit training: fp32 norms/embeddings, enable input grads.
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=mc["gradient_checkpointing"]
    )

    lora_cfg = LoraConfig(
        r=mc["best_rank"], lora_alpha=16, lora_dropout=0.05,
        target_modules=mc["target_modules"], bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)

    trainable, total = 0, 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad: trainable += p.numel()
    print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)")

    train_ds = PromptAnswerDataset(tr_df, cfg, tokenizer)
    train_loader = DataLoader(
        train_ds, batch_size=mc["batch_size"], shuffle=True,
        collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id),
    )

    # Supervisor's exact ask: paged_adamw_8bit
    optim = bnb.optim.PagedAdamW8bit(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=mc["best_lr"],
    )
    total_steps = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps,
    )

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            optim.step(); sched.step(); optim.zero_grad()
            epoch_loss += loss.item()
            if (step + 1) % 100 == 0:
                print(f"  epoch {epoch+1} step {step+1}/{len(train_loader)}  loss={loss.item():.4f}  ({time.time()-t0:.0f}s)")
        print(f"  epoch {epoch+1}: mean_loss={epoch_loss/len(train_loader):.4f}")
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == "cuda" else 0

    print("Evaluating on test set...")
    metrics = eval_model(model, tokenizer, dataset_name)

    if save_adapter_to:
        os.makedirs(save_adapter_to, exist_ok=True)
        model.save_pretrained(save_adapter_to)
        tokenizer.save_pretrained(save_adapter_to)
        print(f"Adapter saved → {save_adapter_to}")

    row = {
        "model": model_id, "dataset": dataset_name, "task": cfg.task,
        "quantisation":   "nf4_double",
        "optimizer":      "paged_adamw_8bit",
        "lora_rank":      mc["best_rank"],
        "lora_alpha":     16,
        "lora_lr":        mc["best_lr"],
        "epochs":         epochs,
        "trainable_params": trainable,
        "total_params":     total,
        "trainable_pct":    round(100*trainable/total, 4),
        "peak_mem_mb":      round(peak_mem_mb, 1),
        "train_time_s":     round(train_time, 1),
        "metric_primary":   metrics["primary_metric"],
        "value_primary":    round(metrics["primary_value"],   4),
        "metric_secondary": metrics["secondary_metric"],
        "value_secondary":  round(metrics["secondary_value"], 4),
    }
    del model, optim, sched, train_loader, train_ds
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return row

print("QLoRA harness ready.")

## 7. Scoreboard helper

In [ ]:
QLORA_COLS = [
    "model", "dataset", "task", "quantisation", "optimizer",
    "lora_rank", "lora_alpha", "lora_lr", "epochs",
    "trainable_params", "total_params", "trainable_pct",
    "peak_mem_mb", "train_time_s",
    "metric_primary", "value_primary",
    "metric_secondary", "value_secondary",
]

def append_qlora_row(row):
    new_df = pd.DataFrame([row])
    if os.path.exists(QLORA_CSV):
        existing = pd.read_csv(QLORA_CSV)
        key = ["model", "dataset", "lora_rank", "lora_lr"]
        existing = existing[~existing[key].apply(tuple, axis=1).isin(
            new_df[key].apply(tuple, axis=1)
        )]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined[QLORA_COLS]
    combined.to_csv(QLORA_CSV, index=False)
    print(f"QLoRA scoreboard now: {len(combined)} rows.")

## 8. Driver — run all 9 (3 models × 3 datasets)

In [ ]:
all_rows = []
for model_id in MODEL_CONFIGS:
    short = MODEL_CONFIGS[model_id]["short_name"]
    for ds_name in ["BBBP", "BACE", "ESOL"]:
        adapter_dir = f"{ADAPTERS_DIR}/qlora_{short}_{ds_name.lower()}"
        print(f"\n{'#'*64}")
        print(f"# QLoRA  {model_id}  |  {ds_name}")
        print(f"#  rank={MODEL_CONFIGS[model_id]['best_rank']}  lr={MODEL_CONFIGS[model_id]['best_lr']:.0e}")
        print(f"{'#'*64}")
        row = qlora_train_eval(model_id, ds_name, save_adapter_to=adapter_dir)
        all_rows.append(row)
        append_qlora_row(row)

print("\n--- All QLoRA runs complete ---")
print(pd.DataFrame(all_rows)[["model","dataset","value_primary","value_secondary","peak_mem_mb","train_time_s"]].to_string(index=False))

## 9. Final QLoRA scoreboard view

In [ ]:
final = pd.read_csv(QLORA_CSV)
print(f"QLoRA scoreboard rows: {len(final)}\n")
print(final.to_string(index=False))

## 10. Done

**Added:** 9 QLoRA rows (3 models × 3 datasets), 9 adapters saved to Drive.

**What we have at this point:**
```
Phase 3 baselines.csv      : 6 rows
Phase 4 baselines.csv      : 27 prompting rows  (33 total)
Phase 5 lora_results.csv   : 18 LoRA rows
Phase 6 qlora_results.csv  : 9 QLoRA rows       ← NEW
TOTAL                      : 60 numbered results
```

**Next: 06b — LoRA vs QLoRA comparison.** Loads both CSVs and produces the supervisor's exact ask (line 97):
- Δ accuracy
- Δ peak memory
- Δ training wall-clock
- Δ inference latency (per-prompt forward pass, timed)